In [ ]:
from google.colab import drive
drive.mount('/content/drive')


import pandas as pd
import sqlite3

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

df=pd.read_csv("/content/drive/MyDrive/Consumo inteligente hackathon/data/processed_dados tratados/customer_features_cluster_ICI.csv")
df.head()

,CustomerID,Recency,Frequency,Monetary,TotalQuantity,AvgUnitPrice,UniqueProducts,CountryCount,AvgInvoiceTicket,MaxInvoiceTicket,AvgSpendPerPurchase,AvgQuantityPerPurchase,Cluster,ICI,Recomendacao,ClusterAvgPrice,PriceIndex
0,12346.0,326,1,77183.60,74215,1.040000,1,1,77183.600000,77183.60,77183.600000,74215.000000,1,24.07,"Consumo crítico. Priorizar revisão de gastos, ...",80709.925000,0.000013
1,12347.0,2,7,4310.00,2458,2.644011,103,1,615.714286,1294.32,615.714286,351.142857,0,79.93,Bom padrão de consumo. Monitorar categorias co...,391.885957,0.006747
2,12348.0,75,4,1797.24,2341,5.764839,22,1,449.310000,892.80,449.310000,585.250000,0,74.04,Bom padrão de consumo. Monitorar categorias co...,391.885957,0.014711
3,12349.0,19,1,1757.55,631,8.289041,73,1,1757.550000,1757.55,1757.550000,631.000000,0,77.88,Bom padrão de consumo. Monitorar categorias co...,391.885957,0.021152
4,12350.0,310,1,334.40,197,3.841176,17,1,334.400000,334.40,334.400000,197.000000,2,55.02,Consumo moderado. Revisar frequência de compra...,327.006549,0.011746


In [ ]:
#Conferindo colunas

df.info()
df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4338 entries, 0 to 4337
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   CustomerID              4338 non-null   float64
 1   Recency                 4338 non-null   int64  
 2   Frequency               4338 non-null   int64  
 3   Monetary                4338 non-null   float64
 4   TotalQuantity           4338 non-null   int64  
 5   AvgUnitPrice            4338 non-null   float64
 6   UniqueProducts          4338 non-null   int64  
 7   CountryCount            4338 non-null   int64  
 8   AvgInvoiceTicket        4338 non-null   float64
 9   MaxInvoiceTicket        4338 non-null   float64
 10  AvgSpendPerPurchase     4338 non-null   float64
 11  AvgQuantityPerPurchase  4338 non-null   float64
 12  Cluster                 4338 non-null   int64  
 13  ICI                     4338 non-null   float64
 14  Recomendacao            4338 non-null   

Index(['CustomerID', 'Recency', 'Frequency', 'Monetary', 'TotalQuantity',
       'AvgUnitPrice', 'UniqueProducts', 'CountryCount', 'AvgInvoiceTicket',
       'MaxInvoiceTicket', 'AvgSpendPerPurchase', 'AvgQuantityPerPurchase',
       'Cluster', 'ICI', 'Recomendacao', 'ClusterAvgPrice', 'PriceIndex'],
      dtype='object')

In [ ]:
import sqlite3

#Criando Banco

conn=sqlite3.connect("/content/drive/MyDrive/Consumo inteligente hackathon/data/processed_dados tratados/Consumo_inteligente.db")
df.to_sql("Clientes_consumo", conn, if_exists="replace", index=False)

print("Tabela criada com sucess!")

Tabela criada com sucess!


In [ ]:
#Conferindo taBela criada

query = """
SELECT * FROM Clientes_consumo
LIMIT 5;
"""
pd.read_sql(query, conn)

,CustomerID,Recency,Frequency,Monetary,TotalQuantity,AvgUnitPrice,UniqueProducts,CountryCount,AvgInvoiceTicket,MaxInvoiceTicket,AvgSpendPerPurchase,AvgQuantityPerPurchase,Cluster,ICI,Recomendacao,ClusterAvgPrice,PriceIndex
0,12346.0,326,1,77183.60,74215,1.040000,1,1,77183.600000,77183.60,77183.600000,74215.000000,1,24.07,"Consumo crítico. Priorizar revisão de gastos, ...",80709.925000,0.000013
1,12347.0,2,7,4310.00,2458,2.644011,103,1,615.714286,1294.32,615.714286,351.142857,0,79.93,Bom padrão de consumo. Monitorar categorias co...,391.885957,0.006747
2,12348.0,75,4,1797.24,2341,5.764839,22,1,449.310000,892.80,449.310000,585.250000,0,74.04,Bom padrão de consumo. Monitorar categorias co...,391.885957,0.014711
3,12349.0,19,1,1757.55,631,8.289041,73,1,1757.550000,1757.55,1757.550000,631.000000,0,77.88,Bom padrão de consumo. Monitorar categorias co...,391.885957,0.021152
4,12350.0,310,1,334.40,197,3.841176,17,1,334.400000,334.40,334.400000,197.000000,2,55.02,Consumo moderado. Revisar frequência de compra...,327.006549,0.011746


In [ ]:
#Media do ICI por cluster

query = """
SELECT cluster,
ROUND(AVG(ICI),2) AS media_ici,
COUNT (CustomerID) AS total_clientes
FROM Clientes_consumo
GROUP BY cluster
ORDER BY media_ici DESC;
"""
pd.read_sql(query, conn)

,Cluster,media_ici,total_clientes
0,3,78.49,23
1,0,76.83,3225
2,2,60.18,1088
3,1,32.07,2


In [ ]:
#Clientes com ICI crítico

query="""
SELECT
    CustomerID,
    Cluster,
    ICI,
    Monetary,
    Frequency,
    AvgInvoiceTicket
FROM Clientes_consumo
WHERE ICI > 40
ORDER BY ICI ASC
LIMIT 20;
"""
pd.read_sql(query, conn)

,CustomerID,Cluster,ICI,Monetary,Frequency,AvgInvoiceTicket
0,16446.0,1,40.07,168472.50,2,84236.25
1,16754.0,2,49.39,2002.40,1,2002.40
2,18074.0,2,49.81,489.60,1,489.60
3,13370.0,2,49.87,754.87,1,754.87
4,14729.0,2,49.88,313.49,1,313.49
5,15165.0,2,49.89,487.75,1,487.75
6,17968.0,2,49.89,277.35,1,277.35
7,16583.0,2,49.91,233.45,1,233.45
8,17908.0,2,49.91,243.28,1,243.28
9,12791.0,2,49.93,192.60,1,192.60


In [ ]:
#Clientes por gasto total

query = """
SELECT
    CustomerID,
    Cluster,
    Monetary
FROM Clientes_consumo
ORDER BY Monetary DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,CustomerID,Cluster,Monetary
0,14646.0,3,280206.02
1,18102.0,3,259657.30
2,17450.0,3,194550.79
3,16446.0,1,168472.50
4,14911.0,3,143825.06
5,12415.0,3,124914.53
6,14156.0,3,117379.63
7,17511.0,3,91062.38
8,16029.0,3,81024.84
9,12346.0,1,77183.60


In [ ]:
#Ticket médio por cluster

query = """
SELECT
    Cluster,
    ROUND(AVG(AvgInvoiceTicket),2) AS ticket_medio,
    ROUND(AVG(Monetary),2) AS gasto_medio,
    ROUND(AVG(Frequency),2) AS frequencia_media
FROM Clientes_consumo
GROUP BY Cluster
ORDER BY ticket_medio DESC;
"""
pd.read_sql(query, conn)

,Cluster,ticket_medio,gasto_medio,frequencia_media
0,1,80709.93,122828.05,1.50
1,3,1622.09,84505.35,73.13
2,0,391.89,1907.47,4.69
3,2,327.01,524.40,1.58


In [ ]:
#Classificando cliente por ICI

# Consulta SQL para categorizar clientes com base na 'Recomendacao'
# Essa célula conta o número total de clientes, calcula o ICI médio,
# o gasto monetário médio e a frequência média para cada categoria de recomendação.
# Os resultados são agrupados por 'Recomendacao' e ordenados pelo número total de clientes em ordem decrescente.
query="""
SELECT
     Recomendacao,
     COUNT(CustomerID) AS total_clientes,
     ROUND(AVG(ICI),2) AS media_ici,
     ROUND(AVG(Monetary),2) AS gasto_medio,
     ROUND(AVG(Frequency),2) AS frequencia_media
FROM Clientes_consumo
GROUP BY Recomendacao
ORDER BY total_clientes DESC;
"""
pd.read_sql(query, conn)

,Recomendacao,total_clientes,media_ici,gasto_medio,frequencia_media
0,Bom padrão de consumo. Monitorar categorias co...,3522,74.47,1832.11,3.54
1,Consumo moderado. Revisar frequência de compra...,508,55.19,906.78,1.35
2,Consumo eficiente. Manter o padrão atual e aco...,307,80.72,6256.99,17.54
3,"Consumo crítico. Priorizar revisão de gastos, ...",1,24.07,77183.60,1.00


In [ ]:
#Clientes com maior risco de consumo ineficiente

query="""
SELECT
    CustomerID,
    Cluster,
    ICI,
    Recency,
    Frequency,
    Monetary,
    AvgInvoiceTicket
FROM Clientes_consumo
WHERE ICI < 60
ORDER BY ICI ASC
LIMIT 20;
"""
pd.read_sql(query, conn)


,CustomerID,Cluster,ICI,Recency,Frequency,Monetary,AvgInvoiceTicket
0,12346.0,1,24.07,326,1,77183.60,77183.60
1,16446.0,1,40.07,1,2,168472.50,84236.25
2,16754.0,2,49.39,372,1,2002.40,2002.40
3,18074.0,2,49.81,374,1,489.60,489.60
4,13370.0,2,49.87,372,1,754.87,754.87
5,14729.0,2,49.88,374,1,313.49,313.49
6,15165.0,2,49.89,373,1,487.75,487.75
7,17968.0,2,49.89,374,1,277.35,277.35
8,16583.0,2,49.91,374,1,233.45,233.45
9,17908.0,2,49.91,374,1,243.28,243.28
